# Chapter 7 — Episodic Memory

## Setup Instructions

To ensure you have the required dependencies to run this notebook, you'll need to have our `llm-agents-from-scratch` framework installed on the running Jupyter kernel. To do this, you can launch this notebook with the following command while within the project's root directory:

```sh
uv run --with jupyter jupyter lab
```

Alternatively, if you just want to use the published version of `llm-agents-from-scratch` without local development, you can install it from PyPi by uncommenting the cell below.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPi
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama installed on your local machine and have its LLM hosting service running. To download Ollama, follow the instructions found on this page: https://ollama.com/download. After downloading and installing Ollama, you can start a service by opening a terminal and running the command `ollama serve`.

In [4]:
import os, shutil, subprocess, time, urllib.request, urllib.error


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    # Lightning persistent path first, then standard locations
    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh"
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

✓ Ollama already running at http://localhost:11434


## Examples

### Example 1: Constructing an Episode and Printing Format Modes

In [5]:
from llm_agents_from_scratch.data_structures import Task, TaskResult
from llm_agents_from_scratch.data_structures.memory import Episode, EpisodeFormatMode

task = Task(instruction="Compute the Hailstone sequence for 6.")
result = TaskResult(task_id=task.id_, content="6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1")
episode = Episode(
    task=task,
    rollout="mock rollout",
    result=result,
    metadata={"reflection": "Always use the hailstone tool."},
)

print("=== XML (default) ===")
print(episode.format(mode=EpisodeFormatMode.XML))

print()
print("=== CONCAT ===")
print(episode.format(mode=EpisodeFormatMode.CONCAT))

=== XML (default) ===
  <episode>
    <task>Compute the Hailstone sequence for 6.</task>
    <result>6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1</result>
    <reflection>Always use the hailstone tool.</reflection>
    <completed_at>2026-06-07 02:20:34</completed_at>
  </episode>

=== CONCAT ===
task: Compute the Hailstone sequence for 6.
result: 6 → 3 → 10 → 5 → 16 → 8 → 4 → 2 → 1
reflection: Always use the hailstone tool.
completed_at: 2026-06-07 02:20:34
